# Pipeline

One notebook for every analysis. Set `UNIT`, `CORPUS` and `WINDOW` below, then
run the stages you need; each writes a file named after those three settings, so
nothing overwrites anything else.

    UNIT     "phone"    the LabPhon analysis
             "syllable" Jason Shaw's alternative

    WINDOW   "base"     articulatory tokens bounded by the acoustics
             "prop25"   widened by 25% of the token's duration on each side,
                        never less than one articulatory frame
             also prop10, prop50, frame1, fixed50 (see utils/windows.py)

The heavy stages have command-line equivalents in `runs/`, which is what to use
for a batch over several corpora or overnight:

    python runs/train_encoders.py --unit syllable --corpus aemri

See RUNNING.md for the order, what each stage costs, and what has to run where.

In [ ]:
import os, pickle, sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))      # this notebook lives in runs/

from utils.corpora import CORPORA
from utils.paths import RunPaths, repo_root
from utils import pipeline as P

UNIT   = "syllable"
CORPUS = "aemri"
WINDOW = "base"

ROOT   = repo_root()
spec   = CORPORA[CORPUS]
paths  = RunPaths(UNIT, CORPUS, WINDOW, root=ROOT).ensure_dirs()
window = spec.window(WINDOW)        # floor raised to one articulatory frame
DEVICE = P.best_device(prefer_mps=False)

print(paths)
print("window:", window.describe())
print("device:", DEVICE, "| corpus:", spec.notes)

In [ ]:
df     = P.load_dataset(spec, paths.dataset, unit=UNIT)
tokens = P.token_frame(df, unit=UNIT)
print(f"{len(df):,} unit pairs, {len(tokens):,} distinct tokens, "
      f"{df['subject'].nunique()} speakers")

if not window.is_identity:
    # What the widening actually buys. If mean_frames_gained is near zero the
    # analysis is a no-op and there is no point running the rest.
    durations = (tokens["end"] - tokens["start"]).values
    print(P.frames_gained(window, durations, fps=spec.art_fps or 100))
    print(P.window_report(tokens, window, sample=300))

## Stage 1 — surprisal

Lexical: no recordings, no window dependence. Minutes.

In [ ]:
from utils.lexicon import surprisal_table

if not os.path.exists(paths.surprisal):
    surprisal_table(spec, df, root=ROOT, unit=UNIT).to_csv(paths.surprisal, index=False)
pd.read_csv(paths.surprisal).describe()

## Stage 2 — encoders

The expensive stage: one pass over every token's recording per epoch.
Checkpoints every 50 batches and resumes, so it is safe to interrupt.

The acoustic encoder does not depend on the window, so it is trained once per
corpus and reused by every window variant.

In [ ]:
from utils.training import fit_ac_scalers, fit_art_scalers, train_autoencoder

stats = None
if spec.rec_stats:
    stats = P.RecordingStats(spec, paths.video_stats)
    stats.build(tokens["art_f"].unique())
    stats.save()
    print(f"intensity statistics for {len(stats)} recordings")

art_scalers = fit_art_scalers(spec, tokens, window, paths.art_scalers)
ac_scalers  = fit_ac_scalers(tokens, paths.ac_scalers)

In [ ]:
train_autoencoder(spec, tokens, spec.build_art_model(DEVICE), stream="art",
                  ckpt_path=paths.art_ckpt, log_path=paths.art_log,
                  window=window, stats=stats, scalers=art_scalers,
                  device=DEVICE, n_epochs=3)

In [ ]:
if not os.path.exists(paths.ac_ckpt):
    train_autoencoder(spec, tokens, spec.build_ac_model(DEVICE), stream="ac",
                      ckpt_path=paths.ac_ckpt, log_path=paths.ac_log,
                      scalers=ac_scalers, device=DEVICE, n_epochs=3)
else:
    print("acoustic encoder already trained:", os.path.relpath(paths.ac_ckpt, ROOT))

## Stage 3 — latents

Encode every token once. Distinctiveness, entropy and mutual information all
read this file instead of decoding the recordings again.

In [ ]:
art_model, ac_model = P.load_encoders(spec, paths.art_ckpt, paths.ac_ckpt,
                                      device=DEVICE)

latents = P.extract_latents(spec, tokens, art_model, ac_model, unit=UNIT,
                            art_window=window, stats=stats,
                            art_scalers=art_scalers, ac_scalers=ac_scalers,
                            device=DEVICE)
np.savez_compressed(paths.latents, unit=UNIT, window=WINDOW, **latents)
print(f"{len(latents['art']):,} of {len(tokens):,} tokens encoded")

## Stage 4 — distinctiveness

Latent distance between the two members of each pair, row-aligned with the
dataset.

In [ ]:
index, key = P.latent_lookup(latents)
art, ac = latents["art"], latents["ac"]

rows = []
for _, r in df.iterrows():
    i = index.get(key(r["art_f"], r[f"{UNIT}1_start"], r[f"{UNIT}1_end"]))
    j = index.get(key(r["art_f"], r[f"{UNIT}2_start"], r[f"{UNIT}2_end"]))
    rows.append((np.nan, np.nan) if i is None or j is None else
                (float(np.linalg.norm(art[i] - art[j])),
                 float(np.linalg.norm(ac[i] - ac[j]))))

distances = pd.DataFrame(rows, columns=["art_latent_dist", "ac_latent_dist"])
distances.to_csv(paths.distances, index=False)
print(f"{distances.notna().all(axis=1).sum():,}/{len(distances):,} pairs measured")

## Stage 5 — conditional entropy

Cross-validated MDN density estimates, both directions.

In [ ]:
for direction, (X, Y) in (("art_given_ac", (latents["ac"], latents["art"])),
                          ("ac_given_art", (latents["art"], latents["ac"]))):
    entropies, _ = P.conditional_entropy(X, Y, latents["labels"], device=DEVICE)
    with open(paths.entropy(direction), "wb") as f:
        pickle.dump(entropies, f)
    print(f"{direction}: {len(entropies)} units")

## Stage 6 — mutual information and flexibility

In [ ]:
from utils import mi

res = mi.compute_all(latents["art"], latents["ac"],
                     list(latents["labels"]), list(latents["groups"]))
mi.per_phoneme(res).to_csv(paths.mi, index=False)
pd.read_csv(paths.mi).sort_values("flexibility").head(15)

## Stage 7 — combine

One long-format table for the mixed models.

In [ ]:
%run ../runs/combine.py --unit $UNIT --corpus $CORPUS --window $WINDOW --overwrite
pd.read_csv(paths.combined).head()